In [ ]:
import pandas as pd
import numpy as np
import re
import string
from datetime import datetime
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

In [ ]:
# Initialize preprocessing tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


In [ ]:
# ============================================================
# STEP 1: LOAD DATASET
# ============================================================
print("="*60)
print("STEP 1: LOADING DATASET")
print("="*60)

df = pd.read_csv('books_rating.csv')
print(f"✓ Dataset loaded successfully!")
print(f"  Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:\n{df.head()}")


In [ ]:
# ============================================================
# STEP 2: INSPECT DATASET
# ============================================================
print("\n" + "="*60)
print("STEP 2: DATASET INSPECTION")
print("="*60)

print(f"\nDataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:\n{df.describe()}")


In [ ]:
# ============================================================
# STEP 3: RENAME COLUMNS FOR CONSISTENCY
# ============================================================
print("\n" + "="*60)
print("STEP 3: MAPPING COLUMNS")
print("="*60)

# Rename to standard column names
df = df.rename(columns={
    'Id': 'item_id',
    'User_id': 'user_id',
    'review/score': 'rating',
    'review/text': 'review_text',
    'review/time': 'timestamp',
    'Title': 'item_title'
})

print(f"✓ Columns renamed for consistency")
print(f"  item_id: Book ID")
print(f"  user_id: User ID")
print(f"  rating: Review score")
print(f"  review_text: Review content")
print(f"  timestamp: Review date")
print(f"  item_title: Book title")

In [ ]:
# ============================================================
# STEP 4: PREPROCESS REVIEWS
# ============================================================
print("\n" + "="*60)
print("STEP 4: PREPROCESSING REVIEWS")
print("="*60)

def clean_review(text):
    """Clean and preprocess review text"""
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove special characters and digits
    text = re.sub(f'[{re.escape(string.punctuation)}0-9]', ' ', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens 
             if token not in stop_words and len(token) > 2]
    
    return ' '.join(tokens)

print("Cleaning and preprocessing reviews...")
df['review_text_processed'] = df['review_text'].apply(clean_review)
print(f"✓ Reviews cleaned and tokenized")


In [ ]:
# ============================================================
# STEP 5: DATA CLEANING
# ============================================================
print("\n" + "="*60)
print("STEP 5: DATA CLEANING")
print("="*60)

initial_len = len(df)

# Remove duplicates (same user reviewing same item)
df = df.drop_duplicates(subset=['user_id', 'item_id'])
print(f"✓ Removed {initial_len - len(df)} duplicate user-item pairs")

# Remove rows with missing critical values
initial_len = len(df)
df = df.dropna(subset=['user_id', 'item_id', 'rating', 'review_text'])
print(f"✓ Removed {initial_len - len(df)} rows with missing critical values")

# Remove reviews that become empty after preprocessing
initial_len = len(df)
df = df[df['review_text_processed'].str.len() > 0]
print(f"✓ Removed {initial_len - len(df)} reviews that became empty after preprocessing")

# Validate and convert ratings to numeric
initial_len = len(df)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[df['rating'].between(1, 5)]
print(f"✓ Validated ratings (1-5 scale): {initial_len - len(df)} rows removed")

# Convert IDs to string
df['user_id'] = df['user_id'].astype(str)
df['item_id'] = df['item_id'].astype(str)
print(f"✓ Converted IDs to string format")

# Parse timestamps
if 'timestamp' in df.columns:
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        initial_len = len(df)
        df = df[df['timestamp'].notna()]
        print(f"✓ Parsed timestamps: {initial_len - len(df)} rows with invalid timestamps removed")
    except:
        print("⚠ Could not parse timestamps, keeping as is")


In [ ]:
# ============================================================
# STEP 6: SELECT CORE COLUMNS FOR ANALYSIS
# ============================================================
print("\n" + "="*60)
print("STEP 6: EXTRACTING METADATA")
print("="*60)

# Keep only essential columns for recommendation
core_columns = ['user_id', 'item_id', 'rating', 'review_text', 'review_text_processed', 'timestamp']
df = df[core_columns]

print(f"✓ Core columns extracted: {df.columns.tolist()}")
print(f"✓ Metadata structure ready")


In [ ]:

# ============================================================
# STEP 7: DATA SUMMARY STATISTICS
# ============================================================
print("\n" + "="*60)
print("STEP 7: PREPROCESSED DATA SUMMARY")
print("="*60)

print(f"\nTotal reviews: {len(df)}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"Unique items: {df['item_id'].nunique()}")
print(f"\nRating distribution:")
print(df['rating'].value_counts().sort_index())
print(f"\nAverage review length: {df['review_text_processed'].str.split().str.len().mean():.0f} words")
print(f"Min review length: {df['review_text_processed'].str.split().str.len().min():.0f} words")
print(f"Max review length: {df['review_text_processed'].str.split().str.len().max():.0f} words")

# Calculate sparsity
total_possible = df['user_id'].nunique() * df['item_id'].nunique()
sparsity = 1 - len(df) / total_possible
print(f"\nMatrix sparsity: {sparsity:.4f} ({sparsity*100:.2f}%)")


In [ ]:
# ============================================================
# STEP 8: SAVE PROCESSED DATA
# ============================================================
print("\n" + "="*60)
print("STEP 8: SAVING PROCESSED DATA")
print("="*60)

df.to_csv('books_rating_processed.csv', index=False)
print(f"✓ Full dataset saved to 'books_rating_processed.csv'")

In [ ]:
# ============================================================
# STEP 9: CREATE TRAIN/VAL/TEST SPLITS
# ============================================================
print("\n" + "="*60)
print("STEP 9: CREATING DATA SPLITS")
print("="*60)

# Shuffle data
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Create splits: 80% train, 10% val, 10% test
n = len(df_shuffled)
train_idx = int(n * 0.8)
val_idx = train_idx + int(n * 0.1)

train_data = df_shuffled[:train_idx]
val_data = df_shuffled[train_idx:val_idx]
test_data = df_shuffled[val_idx:]

print(f"Train set: {len(train_data)} reviews (80%)")
print(f"Validation set: {len(val_data)} reviews (10%)")
print(f"Test set: {len(test_data)} reviews (10%)")

# Save splits
train_data.to_csv('train_data.csv', index=False)
val_data.to_csv('val_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)

print(f"\n✓ Splits saved:")
print(f"  • train_data.csv")
print(f"  • val_data.csv")
print(f"  • test_data.csv")

In [ ]:
# ============================================================
# STEP 10: FINAL SAMPLE AND PREVIEW
# ============================================================
print("\n" + "="*60)
print("STEP 10: FINAL DATA PREVIEW")
print("="*60)

print("\nSample of processed data:")
print(train_data[['user_id', 'item_id', 'rating', 'review_text', 'review_text_processed']].head(3))

print("\n" + "="*60)
print("PHASE 1 COMPLETE!")
print("="*60)
print("\nGenerated files:")
print("  ✓ books_rating_processed.csv (full dataset)")
print("  ✓ train_data.csv (training set)")
print("  ✓ val_data.csv (validation set)")
print("  ✓ test_data.csv (test set)")
print("\nReady for Phase 2: Text Feature Extraction")